In [ ]:
%pip install astropy netCDF4 matplotlib rasterio pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import netCDF4 as nc
import numpy as np
import rasterio
from rasterio.transform import Affine
import pandas as pd

nc_file = "tongariro-after-Copy of EMIT_L2A_RFL_001_20251031T233401_2530415_019.nc"
output_tif = nc_file[:-3] + "_GEOREF.tif"

variable = "reflectance"
fill_value = -9999.0

ds = nc.Dataset(nc_file)

In [3]:
# Save lookup table for band no# -> wavelengths
print(ds.groups["sensor_band_parameters"])
sbp = ds.groups["sensor_band_parameters"]
wavelengths = np.array(sbp.variables["wavelengths"][:])
fwhm = np.array(sbp.variables["fwhm"][:])
good = np.array(sbp.variables["good_wavelengths"][:]).astype(bool)
for name in ["wavelengths", "fwhm", "good_wavelengths"]:
    var = sbp.variables[name]
    print(name, {attr: getattr(var, attr) for attr in var.ncattrs()})
band_params = pd.DataFrame({
    "band": np.arange(1, len(wavelengths) + 1),  # rasterio/GIS band numbers are 1-based
    "wavelength_center": wavelengths,
    "fwhm": fwhm,
    "good_wavelength": good
})

band_params.to_csv("instrument_band_parameters.csv", index=False)


<class 'netCDF4.Group'>
group /sensor_band_parameters:
    dimensions(sizes): 
    variables(dimensions): float32 wavelengths(bands), float32 fwhm(bands), uint8 good_wavelengths(bands)
    groups: 
wavelengths {'_FillValue': np.float32(-9999.0), 'long_name': 'Wavelength Centers', 'units': 'nm'}
fwhm {'_FillValue': np.float32(-9999.0), 'long_name': 'Full Width at Half Max', 'units': 'nm'}
good_wavelengths {'_FillValue': np.uint8(241), 'long_name': 'Wavelengths where reflectance is useable: 1 = good data, 0 = bad data', 'units': 'unitless'}


In [4]:
# Save GeoTIFF
data = ds.variables[variable][:]
data = np.ma.filled(data, fill_value).astype(np.float32)

loc = ds.groups["location"]

# Fill masked / missing GLT cells with EMIT GLT nodata = 0
glt_x = np.ma.filled(loc.variables["glt_x"][:], 0).astype(np.int64)
glt_y = np.ma.filled(loc.variables["glt_y"][:], 0).astype(np.int64)

# Detect data layout
lat = loc.variables["lat"][:]

if data.shape[1:] == lat.shape:
    # (bands, downtrack, crosstrack)
    bands_first = True
    n_bands = data.shape[0]
    y_limit = data.shape[1]
    x_limit = data.shape[2]
elif data.shape[:-1] == lat.shape:
    # (downtrack, crosstrack, bands)
    bands_first = False
    n_bands = data.shape[2]
    y_limit = data.shape[0]
    x_limit = data.shape[1]
else:
    raise ValueError(f"Weird shape: data={data.shape}, lat={lat.shape}")

out_height, out_width = glt_x.shape

# Validate while GLT is still 1-based.
# EMIT GLT nodata is 0, so valid cells must be > 0.
valid = (
    (glt_x > 0) & (glt_y > 0) &
    (glt_x <= x_limit) &
    (glt_y <= y_limit)
)

print("Output grid:", out_height, out_width)
print("Valid GLT pixels:", np.count_nonzero(valid), "of", valid.size)
print("Gap pixels:", valid.size - np.count_nonzero(valid))

# Convert only valid GLT cells to 0-based source indices
src_x = glt_x[valid] - 1
src_y = glt_y[valid] - 1

out = np.full((n_bands, out_height, out_width), fill_value, dtype=np.float32)

for b in range(n_bands):
    if bands_first:
        out[b, valid] = data[b, src_y, src_x]
    else:
        out[b, valid] = data[src_y, src_x, b]

# Use the EMIT geotransform, not min/max raw lat/lon
gt_raw = ds.getncattr("geotransform")
if isinstance(gt_raw, str):
    gt = [float(x) for x in gt_raw.replace(",", " ").split()]
else:
    gt = list(gt_raw)

transform = Affine.from_gdal(*gt)

# Use -9999 nodata rather than NaN for broader GeoTIFF compatibility
with rasterio.open(
    output_tif,
    "w",
    driver="GTiff",
    height=out_height,
    width=out_width,
    count=n_bands,
    dtype="float32",
    crs="EPSG:4326",
    transform=transform,
    nodata=fill_value,
    BIGTIFF="YES"
) as dst:
    dst.write(out)

print("GeoTIFF ready:", output_tif)
ds.close()

Output grid: 2017 2344
Valid GLT pixels: 2428550 of 4727848
Gap pixels: 2299298
GeoTIFF ready: tongariro-after-Copy of EMIT_L2A_RFL_001_20251031T233401_2530415_019_GEOREF.tif


In [5]:
# Check ouput file works
with rasterio.open(output_tif) as src:
    print("CRS:", src.crs)
    print("Transform:", src.transform)
    print("Bounds:", src.bounds)
    print("Size:", src.width, src.height)
    print("Bands:", src.count)
    print("Nodata:", src.nodata)

    test_band = min(63, src.count)
    arr = src.read(test_band, masked=True)
    print("Valid pixels in test band:", arr.count())
    print("Min/max:", arr.min(), arr.max())

CRS: EPSG:4326
Transform: | 0.00, 0.00, 174.69|
| 0.00,-0.00,-38.76|
| 0.00, 0.00, 1.00|
Bounds: BoundingBox(left=174.688605918392, bottom=-39.85332952792009, right=175.95959894587293, top=-38.759646534563)
Size: 2344 2017
Bands: 285
Nodata: -9999.0
Valid pixels in test band: 2367887
Min/max: 0.013635866 1.2292231
